## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 11 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.


---

**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os
import shutil
import time
import pandas as pd
from datetime import datetime

# Inisialisasi SparkSession Tugas
spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# A. Menyiapkan Streaming Source
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

skema_tugas = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

df_tugas = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas)

26/11/12 00:04:01 WARN Utils: Your hostname, aramsamsam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/12 00:04:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 00:04:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


---
**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [4]:
df_tugas_with_total = df_tugas.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_tugas_with_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("grand_total_pendapatan"),
    count("pesanan_id").alias("total_pesanan")
)

query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

---
**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.


In [7]:
df_tugas2 = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu = df_tugas2.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan_window")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

---
**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [8]:
# D. Pengujian dengan 3 Batch Data Simulasi
waktu_now = datetime.now()

# Batch 1
pd.DataFrame({
    "pesanan_id": ["P101", "P102"],
    "nama_cabang": ["Cabang Magelang", "Cabang Semarang"],
    "nama_menu": ["Kopi Tubruk", "Es Kopi Susu"],
    "qty": [2, 3],
    "harga_satuan": [15000, 20000],
    "waktu_pesan": [waktu_now.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

time.sleep(6)
print("=== TUGAS D: AGREGASI CABANG (AFTER BATCH 1) ===")
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 2
pd.DataFrame({
    "pesanan_id": ["P103", "P104"],
    "nama_cabang": ["Cabang Magelang", "Cabang Solo"],
    "nama_menu": ["Americano", "Matcha Latte"],
    "qty": [1, 2],
    "harga_satuan": [18000, 25000],
    "waktu_pesan": [waktu_now.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

time.sleep(6)
print("=== TUGAS D: AGREGASI CABANG (AFTER BATCH 2) ===")
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 3
pd.DataFrame({
    "pesanan_id": ["P105", "P106"],
    "nama_cabang": ["Cabang Semarang", "Cabang Solo"],
    "nama_menu": ["Roti Bakar", "Es Kopi Susu"],
    "qty": [4, 1],
    "harga_satuan": [12000, 20000],
    "waktu_pesan": [waktu_now.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

time.sleep(6)
print("=== TUGAS D: AGREGASI CABANG (AFTER BATCH 3) ===")
spark.sql("SELECT * FROM agregasi_cabang").show()

print("=== TUGAS C: AGREGASI WINDOW 2 MENIT ===")
spark.sql("SELECT window.start, window.end, total_pendapatan_window FROM agregasi_waktu").show(truncate=False)

=== TUGAS D: AGREGASI CABANG (AFTER BATCH 1) ===
+-----------+----------------------+-------------+
|nama_cabang|grand_total_pendapatan|total_pesanan|
+-----------+----------------------+-------------+
+-----------+----------------------+-------------+



=== TUGAS D: AGREGASI CABANG (AFTER BATCH 2) ===


+---------------+----------------------+-------------+
|    nama_cabang|grand_total_pendapatan|total_pesanan|
+---------------+----------------------+-------------+
|Cabang Magelang|                 30000|            1|
|Cabang Semarang|                 60000|            1|
+---------------+----------------------+-------------+



=== TUGAS D: AGREGASI CABANG (AFTER BATCH 3) ===
+---------------+----------------------+-------------+
|    nama_cabang|grand_total_pendapatan|total_pesanan|
+---------------+----------------------+-------------+
|Cabang Magelang|                 48000|            2|
|    Cabang Solo|                 70000|            2|
|Cabang Semarang|                108000|            2|
+---------------+----------------------+-------------+

=== TUGAS C: AGREGASI WINDOW 2 MENIT ===
+-------------------+-------------------+-----------------------+
|start              |end                |total_pendapatan_window|
+-------------------+-------------------+-----------------------+
|2026-11-12 00:10:00|2026-11-12 00:12:00|226000                 |
+-------------------+-------------------+-----------------------+



---

**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [9]:
# E. Sink Berbasis Berkas (Parquet Sink)
folder_output_parquet = "tugas10_output"
folder_checkpoint_parquet = "tugas10_checkpoint"
shutil.rmtree(folder_output_parquet, ignore_errors=True)
shutil.rmtree(folder_checkpoint_parquet, ignore_errors=True)

query_parquet = df_tugas_with_total.writeStream \
    .format("parquet") \
    .option("path", folder_output_parquet) \
    .option("checkpointLocation", folder_checkpoint_parquet) \
    .outputMode("append") \
    .start()

# Kirim Batch 4 untuk menguji Parquet sink
pd.DataFrame({
    "pesanan_id": ["P107"],
    "nama_cabang": ["Cabang Magelang"],
    "nama_menu": ["Affogato"],
    "qty": [2],
    "harga_satuan": [22000],
    "waktu_pesan": [waktu_now.strftime("%Y-%m-%d %H:%M:%S")]
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

time.sleep(6)

# Verifikasi Pembacaan Parquet File
df_parquet_verify = spark.read.parquet(folder_output_parquet)
print("=== TUGAS E: VERIFIKASI FILE PARQUET TERBALIK DIBACA ===")
df_parquet_verify.show()

# Penghentian Query
query_cabang.stop()
query_waktu.stop()
query_parquet.stop()
spark.stop()
print("Seluruh streaming query dihentikan & SparkSession ditutup.")

=== TUGAS E: VERIFIKASI FILE PARQUET TERBALIK DIBACA ===
+----------+---------------+------------+---+------------+-------------------+-----------+
|pesanan_id|    nama_cabang|   nama_menu|qty|harga_satuan|        waktu_pesan|total_harga|
+----------+---------------+------------+---+------------+-------------------+-----------+
|      P101|Cabang Magelang| Kopi Tubruk|  2|       15000|2026-11-12 00:11:09|      30000|
|      P102|Cabang Semarang|Es Kopi Susu|  3|       20000|2026-11-12 00:11:09|      60000|
|      P107|Cabang Magelang|    Affogato|  2|       22000|2026-11-12 00:11:09|      44000|
|      P105|Cabang Semarang|  Roti Bakar|  4|       12000|2026-11-12 00:11:09|      48000|
|      P106|    Cabang Solo|Es Kopi Susu|  1|       20000|2026-11-12 00:11:09|      20000|
|      P103|Cabang Magelang|   Americano|  1|       18000|2026-11-12 00:11:09|      18000|
|      P104|    Cabang Solo|Matcha Latte|  2|       25000|2026-11-12 00:11:09|      50000|
+----------+---------------+-----